<a href="https://colab.research.google.com/github/MelinaNk/PhD/blob/main/NC_files_DTMK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%%script echo skipping
!pip install xarray netcdf4
!pip install python-docx
!pip install --upgrade netCDF4 h5py


from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import xarray as xr
import pandas as pd
import os
import numpy as np

Location="Ardasa_Kozani"
longt=21.64
latit=40.48
# Set the working directory
os.chdir('/content/drive/MyDrive/Comparisons_DTMK')

# Create a list to store dataframes for each file
dfs = []

# Folder containing NetCDF files
file_folder = '/content/drive/MyDrive/Comparisons_DTMK/ORD55703'
os.makedirs(file_folder, exist_ok=True)
# List all csv files in the directory
nc_files = [file for file in os.listdir(file_folder) if (file.endswith('.nc') and file.startswith('SISdm20'))]
# Iterate through each NetCDF file in the folder
for i, file_name in enumerate(nc_files):

  file_path = os.path.join(file_folder, file_name)
  try:
      # Open the NetCDF file using xarray
      ds = xr.open_dataset(file_path)

      # Round 'lon' and 'lat' coordinates to 3 and 2 decimals, respectively
      rounded_lon = ds.lon.round(3)
      rounded_lat = ds.lat.round(2)

      # Find the nearest indices for the given latitude and longitude
      lon_index = abs(rounded_lon - longt).argmin().item()
      lat_index = abs(rounded_lat - latit).argmin().item()

      # Extract values for the specified location using method='nearest'
      sis_values = ds['SIS'].isel(lon=lon_index, lat=lat_index).values
      sisc_values = ds['SISC'].isel(lon=lon_index, lat=lat_index).values

      # Extract year, month, and day from the file name
      year = int(file_name[5:9])
      month = int(file_name[9:11])
      day = int(file_name[11:13])

      # Create a DataFrame for the current file
      df = pd.DataFrame({
          'Year': year,
          'Month': month,
          'Day': day,
          'SIS': sis_values.flatten(),
          'SISC': sisc_values.flatten()
      })

      # Append the DataFrame to the list
      dfs.append(df)
      # Log progress every 10 files
      if (i + 1) % 10 == 0:
          print(f"Processed {i + 1} files out of {len(nc_files)}")

  except Exception as e:
      print(f"Error processing file {file_path}: {e}")

# Concatenate all DataFrames into a single DataFrame
result_df = pd.concat(dfs, ignore_index=True)
print(result_df.head())

# Save the final DataFrame to CSV
output_csv_path = os.path.join(file_folder, f"{Location}_output_data_all.csv")
result_df.to_csv(output_csv_path, index=False)

# Provide the path to the saved CSV file
print(f"Combined CSV file saved to: {output_csv_path}")




Processed 10 files out of 1156


In [ ]:
import pandas as pd

# Read the CSV file
file_path = '/content/drive/MyDrive/Comparisons_DTMK/ORD55703/'+Location+'output_data_all.csv'
df = pd.read_csv(file_path)

# Sort the DataFrame by 'Year', 'Month', and 'Day' columns
df_sorted = df.sort_values(by=['Year', 'Month', 'Day'])

# Write the sorted DataFrame back to the same file
df_sorted.to_csv(file_path, index=False)

print("File sorted and replaced successfully.")

In [ ]:
%%script echo skipping
import pandas as pd
import os

# Set the working directory
os.chdir('/content/drive/MyDrive/Comparisons_DTMK')

# Folder paths
folder1 = '/content/drive/MyDrive/Comparisons_DTMK/ORD54426'
folder2 = '/content/drive/MyDrive/Comparisons_DTMK/ORD54427'
folder3 = '/content/drive/MyDrive/Comparisons_DTMK/ORD54537'

# File paths
file1 = os.path.join(folder1, Location+'output_data_all.csv')
file2 = os.path.join(folder2, Location+'output_data_all.csv')
file3 = os.path.join(folder3, Location+'output_data_all.csv')

# Read both CSV files while excluding the 'SISC' column
df1 = pd.read_csv(file1).drop(columns=['SISC'])
df2 = pd.read_csv(file2).drop(columns=['SISC'])
df3 = pd.read_csv(file3).drop(columns=['SISC'])

# Multiply the 'SIS' column by 0.0864
df1['SIS'] *= 0.0864
df2['SIS'] *= 0.0864
df3['SIS'] *= 0.0864

# Concatenate the DataFrames
combined_df = pd.concat([df1, df2,df3], ignore_index=True)

# Sort the combined DataFrame by 'Year', 'Month', and 'Day' columns
combined_df_sorted = combined_df.sort_values(by=['Year', 'Month', 'Day'])
# Write the combined DataFrame to a new CSV file
combined_file = '/content/drive/MyDrive/Comparisons_DTMK/CMF_SRAD/'+Location+'_combined_file.csv'
combined_df.to_csv(combined_file, index=False)

print("Files combined successfully.")


Files combined successfully.
